In [ ]:
import warnings
warnings.filterwarnings("ignore", message=".*Pydantic serializer warnings.*", category=UserWarning)

from dotenv import load_dotenv

load_dotenv()

In [ ]:
from dataclasses import dataclass

@dataclass
class EmailContext:
    email_address: str = "farhan@example.com"
    password: str = "password123"

In [ ]:
from langchain.agents import AgentState

class AuthenticatedState(AgentState):
    authenticated: bool

In [ ]:
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage

@tool
def check_inbox() -> str:
    """Check the inbox for the recent emails"""

    return """
    Hey Farhan I am in the town for a week, How about we meet and discuss business,
    if you can make some time and come for a dinner let me know,
    -best, Julie(julie@example.com)
    """

@tool
def send_email(to: str, subject: str, body: str) -> str:
    """Send a response Email"""

    return f"Email Sent to {to} with subject {subject} body {body}"


@tool
def authenticate(email: str, password: str, runtime: ToolRuntime) -> Command:
    """Authenticate the User with given email and password"""
    if email == runtime.context.email_address and password == runtime.context.password:
        return Command(
            update={
                "Authenticated" : True,
                "messages" : [ToolMessage("Successfully authenticated", tool_call_id=runtime.tool_call_id)],
            }
        )

    else:
        return Command(
            update={
                "Authenticated" : False,
                "messages" : [ToolMessage("Authentication failed", tool_call_id=runtime.tool_call_id)]
            }
        )

In [ ]:
from langchain.agents.middleware import wrap_model_call,ModelRequest,ModelResponse
from typing import Callable

@wrap_model_call
async def dynamic_tool_call(
        request: ModelRequest, handler: Callable[[ModelRequest],ModelResponse]
) -> ModelResponse:
    """Allow read inbox and send Email tools only if user provide correct email and password"""

    authenticated = request.state(authenticate)

    if authenticated:
        tools=[check_inbox,send_email]
    else:
        tools=[authenticate]

    request=request.override(tools=tools)
    return await handler(request)


In [ ]:
from langchain.agents.middleware import dynamic_prompt

authenticated_prompt = """You are a helpful assistant that can check the inbox and send emails.
Your first step after authentication is to check the inbox."""
unauthenticated_prompt = "You are a helpful assistant that can authenticate users."


@dynamic_prompt
def dynamic_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on authentication status"""
    authenticated = request.state.get("authenticated")

    if authenticated:
        return authenticated_prompt
    else:
        return unauthenticated_prompt

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

agent = create_agent(
    "google_genai:gemini-2.5-flash",
    tools=[authenticate, check_inbox, send_email],
    checkpointer=InMemorySaver(),
    state_schema=AuthenticatedState,
    context_schema=EmailContext,
    middleware=[
        dynamic_tool_call,
        dynamic_prompt,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "authenticate": False,
                "check_inbox": False,
                "send_email": True,
            })
    ]
)


In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {"messages": [HumanMessage(content="farhan@example.com, password123")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [ ]:

response = agent.invoke(
    {"messages": [HumanMessage(content="any draft is fine. don't check back.")]},
    context=EmailContext(),
    config=config
)

print(response['messages'][-1].content)

In [ ]:
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

In [ ]:
from langgraph.types import Command

response = agent.invoke(
    Command(
        resume={"decisions": [{"type": "approve"}]}  # or "reject"
    ),
    config=config  # Same thread ID to resume the paused conversation
)

print(response["messages"][-1].content)

In [ ]:
from pprint import pprint

pprint(response)